# QCML for Credit Risk Classification
### Quantum Cognition Machine Learning applied to Banking: UCI German Credit Dataset

**Motivation**

The QCML algorithm (Musaelian et al., Qognitive Inc.) encodes each data point $x \in \mathbb{R}^D$ as the ground state $|\psi_0(x)\rangle$ of a Hamiltonian built from $D$ learned Hermitian observables:

$$H(x) = \sum_{k=1}^{D}(A_k - x_k\,I)^2$$

This construction was designed for high-dimensional structured data. The QCML research group has demonstrated financial applications in corporate bond similarity [Q3] and financial time-series forecasting [Q2]. This notebook extends those ideas to retail credit risk: one of the most important and regulated problems in banking.

**What we test**

| Pipeline | Role |
|---|---|
| Logistic Regression on raw features | Baseline |
| PCA + Logistic Regression | Standard dimensionality-reduction baseline |
| QCML + K-Means (unsupervised) | Unsupervised clustering quality |
| QCML + Logistic Regression | Hybrid: quantum geometry → supervised classifier |

We also use QCML's built-in diagnostics: observable eigenvalue spread and per-feature quantum variance: to identify **which financial features drive the learned geometry**. This addresses a critical requirement in regulated banking: interpretability.

**References**
- [Q2] Samson et al., *QCML: Financial Forecasting*, QognitiveAI 2024
- [Q3] Rosaler et al., *Supervised Similarity for High-Yield Corporate Bonds*, QognitiveAI 2025
- Dua, D. and Graff, C. (2019). UCI ML Repository. German Credit Data.


## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# qcml/ lives in the same directory as this notebook
from qcml.core import QCML
from qcml.utils import (
    best_kmeans,        # used only for RAW and PCA baselines, never on QCML embeddings
    align_labels,
    cluster_accuracy,
    cluster_metrics,
    classification_metrics,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

SEED = 42
np.random.seed(SEED)


# ── QCML intrinsic clustering ────────────────────────────────────────────────
def qcml_cluster_labels(Z: np.ndarray, k: int) -> np.ndarray:
    """
    Assign cluster labels from QCML ground-state embeddings without K-Means.

    Builds the quantum covariance C = Z^T Z, extracts its top-k eigenvectors
    (the dominant quantum modes on the sphere), and assigns each point to the
    mode it projects onto most strongly:

        label(x) = argmax_r |<ψ₀(x) | v_r>|

    Parameters
    ----------
    Z : array (N, m) : unit-norm ground states from model.transform()
    k : int : number of clusters

    Returns
    -------
    labels : array (N,) of integers in {0, ..., k-1}
    """
    C = Z.T @ Z
    _, eigvecs = np.linalg.eigh(C)
    V_k = eigvecs[:, -k:]
    scores = Z @ V_k
    return np.argmax(np.abs(scores), axis=1)


print("Imports OK.")

## 2. Dataset: UCI German Credit

1,000 loan applicants. Each is labelled **good** (creditworthy) or **bad** (default risk).  
20 raw features: a mix of numeric (loan amount, duration, age) and categorical (purpose, employment status, housing).  
After one-hot encoding the categoricals, we get ~60 binary/numeric features: a realistic credit bureau feature set.

In [ ]:
# Load from OpenML (no authentication required)
raw = fetch_openml("credit-g", version=1, as_frame=True, parser="auto")
df  = raw.frame.copy()

# Binary label: good=1, bad=0
df["label"] = (df["class"] == "good").astype(int)
df = df.drop(columns=["class"])

y = df["label"].values
X_raw = df.drop(columns=["label"])

numeric_cols     = X_raw.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_raw.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Samples    : {len(df)}")
print(f"Numeric    : {len(numeric_cols)} features")
print(f"Categorical: {len(categorical_cols)} features → will be one-hot encoded")
print(f"Class balance: good={y.sum()} ({y.mean():.1%}), bad={len(y)-y.sum()} ({1-y.mean():.1%})")
df.head()

## 3. Preprocessing

- Categorical features → one-hot encoding (drop first to avoid collinearity)
- All features → StandardScaler (zero mean, unit variance)
- Stratified 80/20 train/test split

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical_cols),
])

X_proc = preprocessor.fit_transform(X_raw)   # shape (1000, D)
D = X_proc.shape[1]

# Feature names after encoding (for diagnostics later)
cat_encoder    = preprocessor.named_transformers_["cat"]
cat_feat_names = cat_encoder.get_feature_names_out(categorical_cols).tolist()
feature_names  = numeric_cols + cat_feat_names

X_train, X_test, y_train, y_test = train_test_split(
    X_proc, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f"Feature dimension after encoding: D = {D}")
print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

## 4. Baseline 1: Logistic Regression on Raw (Standardized) Features

In [ ]:
lr_raw = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
lr_raw.fit(X_train, y_train)

y_pred_raw  = lr_raw.predict(X_test)
y_prob_raw  = lr_raw.predict_proba(X_test)[:, 1]

acc_raw = accuracy_score(y_test, y_pred_raw)
auc_raw = roc_auc_score(y_test, y_prob_raw)

print("Logistic Regression: raw features")
print(f"  Accuracy : {acc_raw:.4f}")
print(f"  ROC-AUC  : {auc_raw:.4f}")
print()
print(classification_report(y_test, y_pred_raw, target_names=["bad", "good"]))

## 5. Baseline 2: PCA + Logistic Regression

We retain enough components to explain 95% of variance, then classify on the reduced representation.  
This is the standard linear-algebra baseline that QCML is compared against.

In [ ]:
pca = PCA(n_components=0.95, random_state=SEED)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

lr_pca = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
lr_pca.fit(X_train_pca, y_train)

y_pred_pca  = lr_pca.predict(X_test_pca)
y_prob_pca  = lr_pca.predict_proba(X_test_pca)[:, 1]

acc_pca = accuracy_score(y_test, y_pred_pca)
auc_pca = roc_auc_score(y_test, y_prob_pca)

n_components_kept = pca.n_components_

print(f"PCA kept {n_components_kept} components ({pca.explained_variance_ratio_.sum():.1%} variance)")
print(f"PCA + LogReg")
print(f"  Accuracy : {acc_pca:.4f}")
print(f"  ROC-AUC  : {auc_pca:.4f}")

## 6. QCML Training

QCML is trained **unsupervised** on the training features only (no labels used).  
The learned observables $\{A_k\}$ encode the geometry of the credit feature space.  
The output $Z = \{|\psi_0(x_t)\rangle\}$ lives on the unit sphere in $\mathbb{R}^m$.

**Hyperparameters**

| Parameter | Value | Rationale |
|---|---|---|
| `hilbert_dim` m | 32 | Rule of thumb: m ≥ √D. D≈60 → m=32 balances capacity and runtime |
| `w` | 0.25 | Empirically best for linear separability (see ex5_effect_of_w) |
| `lr` | 3e-3 | Default Adam rate, stable across all tested datasets |
| `epochs` | 400 | Extended for financial data; watch for convergence plateau |
| `batch_size` | 32 | Standard mini-batch |


In [ ]:
HILBERT_DIM = 32
W           = 0.25      # variance weight: w=0.25 maximizes linear separability
EPOCHS      = 400
LR          = 3e-3
BATCH       = 32

model = QCML(hilbert_dim=HILBERT_DIM, lr=LR, w=W, seed=SEED)
model.fit(X_train, epochs=EPOCHS, batch_size=BATCH, verbose=True)

## 7. Extract Ground-State Embeddings

In [ ]:
print("Computing ground-state embeddings...")
Z_train = model.transform(X_train)   # shape (N_train, m)
Z_test  = model.transform(X_test)    # shape (N_test,  m)

print(f"Z_train shape: {Z_train.shape}  (each row is a unit vector on S^{HILBERT_DIM-1})")
print(f"Z_test  shape: {Z_test.shape}")
print(f"||Z_train[0]|| = {np.linalg.norm(Z_train[0]):.6f}  (should be 1.0)")

## 8. QCML Intrinsic Clustering (Unsupervised Evaluation)

QCML's ground-state embeddings define natural clusters on the unit sphere : we do **not** apply K-Means to the embeddings, as that would misrepresent the method.

Instead, `qcml_cluster_labels(Z, k)` recovers the clusters by decomposing the quantum covariance $C = Z^\top Z$: the top-$k$ eigenvectors are the dominant quantum modes, and each applicant is assigned to the mode they project onto most strongly. No external clustering algorithm is involved.

In [ ]:
K = 2   # good / bad

# Raw K-Means on training data (external baseline : fine to use K-Means here)
km_train_raw    = best_kmeans(X_train, K)
km_acc_raw      = cluster_accuracy(y_train, km_train_raw, K)
km_metrics_raw  = cluster_metrics(y_train, km_train_raw, X_train)

# QCML intrinsic clustering : no K-Means used
qcml_labels_train  = qcml_cluster_labels(Z_train, K)
km_acc_qcml        = cluster_accuracy(y_train, qcml_labels_train, K)
km_metrics_qcml    = cluster_metrics(y_train, qcml_labels_train, Z_train)

print("K-Means on raw features (train) : external baseline")
print(f"  Accuracy (Hungarian): {km_acc_raw:.4f}")
for k, v in km_metrics_raw.items():
    print(f"  {k:12s}: {v:.4f}")

print(f"\nQCML intrinsic clustering (train, m={HILBERT_DIM}, w={W})")
print(f"  Accuracy (Hungarian): {km_acc_qcml:.4f}")
for k, v in km_metrics_qcml.items():
    print(f"  {k:12s}: {v:.4f}")

## 9. QCML + Logistic Regression (Supervised Pipeline)

Use the **unsupervised** QCML embeddings $Z$ as features for a logistic regression classifier.  
This is the key hybrid pipeline: QCML learns the geometry of the credit space, LogReg exploits it.  
Labels are only seen by the logistic regression: QCML is purely unsupervised.

In [ ]:
lr_qcml = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
lr_qcml.fit(Z_train, y_train)

y_pred_qcml = lr_qcml.predict(Z_test)
y_prob_qcml = lr_qcml.predict_proba(Z_test)[:, 1]

acc_qcml = accuracy_score(y_test, y_pred_qcml)
auc_qcml = roc_auc_score(y_test, y_prob_qcml)

qcml_clf_metrics = classification_metrics(y_test, y_pred_qcml)

print(f"QCML (m={HILBERT_DIM}, w={W}) + Logistic Regression")
print(f"  Accuracy : {acc_qcml:.4f}")
print(f"  ROC-AUC  : {auc_qcml:.4f}")
print(f"  TPR (recall on 'good') : {qcml_clf_metrics['TPR']:.4f}")
print(f"  TNR (recall on 'bad')  : {qcml_clf_metrics['TNR']:.4f}")
print()
print(classification_report(y_test, y_pred_qcml, target_names=["bad", "good"]))

## 10. Feature Diagnostics: What Does QCML Learn About Credit Data?

QCML's built-in diagnostics reveal which financial features are most informative:

- **Observable eigenvalue spread** `std(eig(A_k))`: high spread → A_k spans a wide measurement range → the feature is actively used. Near-zero → feature was effectively ignored.
- **Per-feature quantum variance** `Var_ψ(A_k)`: high variance → the feature creates quantum uncertainty, a proxy for informativeness.
- **Per-feature reconstruction MSE**: how accurately QCML predicts `x_k` from the ground state: low MSE = feature is well-captured by the learned geometry.

For regulated banking, this directly answers: *which features drive my model's decisions?*

In [ ]:
print("Computing feature diagnostics...")

eig_spread   = model.observable_eigenvalue_spread()              # (D,)
q_var        = model.per_feature_quantum_variance(X_train)       # (D,)
recon_mse    = model.per_feature_recon_mse(X_train)              # (D,)

diag_df = pd.DataFrame({
    "feature"    : feature_names,
    "eig_spread" : eig_spread,
    "q_variance" : q_var,
    "recon_mse"  : recon_mse,
}).sort_values("eig_spread", ascending=False).reset_index(drop=True)

print("Top 15 features by observable eigenvalue spread (most used by QCML):")
print(diag_df.head(15).to_string(index=False))

## 11. Spectral Gap Analysis

The spectral gap $\lambda_1(H(x)) - \lambda_0(H(x))$ measures how well-separated the ground state is from the first excited state.

- **Large gap** → the ground-state embedding is robust (small perturbations in $x$ don't flip the embedding significantly)
- **Small gap** → the embedding is fragile at this point; the model is uncertain

In a credit context: applicants with small spectral gaps are the borderline cases where the model is most uncertain: exactly the cases that warrant human review.

In [ ]:
print("Computing spectral gaps on training set (sample of 200)...")
gaps = model.spectral_gaps(X_train, n_samples=200)   # shape (200,)

# Compare gap distributions: good vs. bad applicants in the sample
rng      = np.random.default_rng(0)
idx_200  = rng.choice(len(X_train), size=min(200, len(X_train)), replace=False)
y_sample = y_train[idx_200]

gap_good = gaps[y_sample == 1]
gap_bad  = gaps[y_sample == 0]

print(f"Spectral gap: good applicants: mean={gap_good.mean():.4f}, std={gap_good.std():.4f}")
print(f"Spectral gap: bad  applicants: mean={gap_bad.mean():.4f}, std={gap_bad.std():.4f}")
print(f"Overall: min={gaps.min():.4f}, median={np.median(gaps):.4f}, max={gaps.max():.4f}")

## 12. Visualisations

In [ ]:
fig = plt.figure(figsize=(16, 14))
gs  = gridspec.GridSpec(3, 2, hspace=0.45, wspace=0.35)

# ── Panel 1: Training loss curve ────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(model.loss_history, color="steelblue", linewidth=1.5)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Ground-state energy loss")
ax1.set_title("QCML Training Loss")
ax1.grid(True, alpha=0.3)

# ── Panel 2: PCA of QCML embeddings, coloured by credit label ───────────────
ax2 = fig.add_subplot(gs[0, 1])
pca2 = PCA(n_components=2, random_state=SEED)
Z2   = pca2.fit_transform(Z_train)
sc   = ax2.scatter(Z2[:, 0], Z2[:, 1], c=y_train, cmap="RdYlGn",
                   alpha=0.6, s=18, edgecolors="none")
plt.colorbar(sc, ax=ax2, label="0=bad, 1=good")
ax2.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%} var)")
ax2.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%} var)")
ax2.set_title("QCML Embeddings (PCA projection)")
ax2.grid(True, alpha=0.2)

# ── Panel 3: Observable eigenvalue spread (top 20 features) ─────────────────
ax3 = fig.add_subplot(gs[1, :])
top20 = diag_df.head(20)
bars  = ax3.barh(top20["feature"][::-1], top20["eig_spread"][::-1],
                  color="cornflowerblue", edgecolor="white")
ax3.set_xlabel("Observable eigenvalue spread  std(eig(Aₖ))")
ax3.set_title("Top 20 Financial Features by QCML Observable Spread\n"
              "(high spread = feature actively encoded in quantum geometry)")
ax3.grid(True, axis="x", alpha=0.3)

# ── Panel 4: Spectral gap histogram ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 0])
ax4.hist(gap_good, bins=25, alpha=0.6, color="seagreen",  label="good credit", density=True)
ax4.hist(gap_bad,  bins=25, alpha=0.6, color="tomato",    label="bad credit",  density=True)
ax4.set_xlabel("Spectral gap  λ₁ − λ₀")
ax4.set_ylabel("Density")
ax4.set_title("Spectral Gap Distribution by Credit Class\n"
              "(small gap = model uncertain = borderline applicant)")
ax4.legend()
ax4.grid(True, alpha=0.3)

# ── Panel 5: Confusion matrix (QCML + LogReg) ───────────────────────────────
ax5 = fig.add_subplot(gs[2, 1])
cm  = confusion_matrix(y_test, y_pred_qcml)
disp = ConfusionMatrixDisplay(cm, display_labels=["bad", "good"])
disp.plot(ax=ax5, colorbar=False, cmap="Blues")
ax5.set_title(f"QCML + LogReg\nAcc={acc_qcml:.3f}  AUC={auc_qcml:.3f}")

plt.suptitle("QCML for Credit Risk: German Credit Dataset", fontsize=13, y=1.01)
plt.savefig("qcml_credit_risk_summary.png", bbox_inches="tight", dpi=150)
plt.show()
print("Figure saved: qcml_credit_risk_summary.png")

## 13. Reconstruction Quality

In [ ]:
errs = model.reconstruction_errors(X_train)

print(f"Mean per-sample reconstruction MSE : {errs.mean():.4f}")
print(f"Median                             : {np.median(errs):.4f}")
print(f"Max                                : {errs.max():.4f}")

# Are good applicants better reconstructed than bad?
print()
print(f"Recon MSE: good applicants: {errs[y_train==1].mean():.4f}")
print(f"Recon MSE: bad  applicants: {errs[y_train==0].mean():.4f}")

## 14. Results Summary

Full comparison across all pipelines on the held-out test set.

In [ ]:
summary = pd.DataFrame([
    {
        "Pipeline"   : "LogReg : raw features",
        "Accuracy"   : round(acc_raw, 4),
        "ROC-AUC"    : round(auc_raw, 4),
        "Clustering" : ":",
        "Notes"      : f"D={D} features, external K-Means baseline",
    },
    {
        "Pipeline"   : "PCA + LogReg",
        "Accuracy"   : round(acc_pca, 4),
        "ROC-AUC"    : round(auc_pca, 4),
        "Clustering" : ":",
        "Notes"      : f"{n_components_kept} PCA components (95% var)",
    },
    {
        "Pipeline"   : f"QCML (m={HILBERT_DIM}, w={W}) intrinsic clustering",
        "Accuracy"   : round(km_acc_qcml, 4),
        "ROC-AUC"    : ":",
        "Clustering" : "Intrinsic (top eigenvectors of Z^T Z)",
        "Notes"      : "No labels used during QCML training or clustering",
    },
    {
        "Pipeline"   : f"QCML (m={HILBERT_DIM}, w={W}) + LogReg",
        "Accuracy"   : round(acc_qcml, 4),
        "ROC-AUC"    : round(auc_qcml, 4),
        "Clustering" : "Supervised (LogReg on embeddings)",
        "Notes"      : "QCML unsupervised geometry → supervised classifier",
    },
])

summary.set_index("Pipeline")

## 15. Discussion: What the Ground State Represents for Financial Data

In the QCML framework, the ground state $|\psi_0(x)\rangle$ is the quantum state that best reconciles all $D$ measurement constraints simultaneously. For an applicant $x$:

- Each observable $A_k$ represents a **measurement device** calibrated to feature $k$ (e.g., loan duration, credit amount, employment status)
- The Hamiltonian $H(x) = \sum_k (A_k - x_k I)^2$ encodes the *tension* between all measurements: it is minimized by a state that simultaneously satisfies all feature constraints as well as possible
- The ground state is therefore a **holistic representation** of the applicant that captures feature interactions, not just individual values

**Financial interpretation of eigenvalue spread:**  
A high spread in $\text{eig}(A_k)$ means observable $A_k$ has learned to distinguish between a wide range of values for feature $k$: the algorithm found this feature geometrically important in the credit space. Low spread means the feature is near-redundant given the others.

**Financial interpretation of spectral gap:**  
Applicants with small $\lambda_1 - \lambda_0$ have nearly degenerate ground states: their credit profile sits near a boundary in the learned geometry. In practice these are the **borderline cases** and should be flagged for human review, which is exactly what Canadian banking regulation (OSFI guidelines) requires for high-impact automated decisions.

**Relevance to Scotiabank:**  
These diagnostics address two simultaneous requirements:
1. **Predictive performance**: competitive accuracy and AUC vs. standard baselines
2. **Interpretability**: model-intrinsic feature importance and uncertainty quantification, without requiring post-hoc explanation tools like SHAP

QCML's physics-derived uncertainty measure (the spectral gap) is a candidate for regulatory-grade uncertainty reporting in automated credit decisions.

## 16. Hyperparameter Sweep: Effect of Hilbert Dimension m

Larger $m$ gives the model more expressivity at the cost of compute.  
We sweep $m \in \{8, 16, 24, 32\}$ and record test accuracy and AUC.

In [ ]:
sweep_results = []
m_values = [8, 16, 24, 32]

for m in m_values:
    print(f"Training QCML with m={m}...")
    qcml_m = QCML(hilbert_dim=m, lr=LR, w=W, seed=SEED)
    qcml_m.fit(X_train, epochs=300, batch_size=BATCH, verbose=False)
    Z_tr = qcml_m.transform(X_train)
    Z_te = qcml_m.transform(X_test)
    # Supervised: QCML embeddings → LogReg
    lr_m = LogisticRegression(max_iter=1000, random_state=SEED)
    lr_m.fit(Z_tr, y_train)
    acc_m = accuracy_score(y_test, lr_m.predict(Z_te))
    auc_m = roc_auc_score(y_test, lr_m.predict_proba(Z_te)[:, 1])
    # Unsupervised: QCML intrinsic clustering (no K-Means)
    lbl_m    = qcml_cluster_labels(Z_tr, 2)
    acc_clus = cluster_accuracy(y_train, lbl_m, 2)
    sweep_results.append({
        "m": m,
        "QCML+LogReg Acc": round(acc_m, 4),
        "QCML+LogReg AUC": round(auc_m, 4),
        "QCML Intrinsic Acc": round(acc_clus, 4),
    })
    print(f"  m={m}: LogReg Acc={acc_m:.4f}  AUC={auc_m:.4f}  Intrinsic Acc={acc_clus:.4f}")

sweep_df = pd.DataFrame(sweep_results)
print()
print(sweep_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric in zip(axes, ["Accuracy", "ROC-AUC"]):
    ax.plot(sweep_df["m"], sweep_df[metric], marker="o", color="steelblue", linewidth=2)
    # Reference lines for baselines
    ref = acc_raw if metric == "Accuracy" else auc_raw
    ax.axhline(ref, color="tomato",   linestyle="--", label=f"LogReg raw ({ref:.3f})")
    ref2 = acc_pca if metric == "Accuracy" else auc_pca
    ax.axhline(ref2, color="darkorange", linestyle=":", label=f"PCA+LogReg ({ref2:.3f})")
    ax.set_xlabel("Hilbert dimension m")
    ax.set_ylabel(metric)
    ax.set_title(f"QCML + LogReg: {metric} vs. m")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(m_values)

plt.tight_layout()
plt.savefig("qcml_hilbert_sweep.png", bbox_inches="tight", dpi=150)
plt.show()